# Lab 3.2 — Building a FastAPI ML Prediction Service
**Module 3: Bridging ML and Engineering**

In this lab you will:
- Understand the transition from Jupyter Notebook → deployable Python service
- Design a **FastAPI** application with Pydantic request/response schemas
- Build `/health`, `/predict`, and `/predict/batch` endpoints
- Test the API in-notebook using `TestClient` (no running server required)
- Run the service locally with `uvicorn` and send live requests

> **Instructor Note:** In production, no one calls `model.predict()` from a notebook. The standard pattern is: train in a notebook → serialize the model → wrap in an API → containerise → deploy. This lab covers steps 3–4. The key insight is that FastAPI + Pydantic gives you automatic input validation and auto-generated Swagger docs for free.


## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| fastapi | `fastapi` |
| uvicorn | `uvicorn[standard]` |
| pydantic | `pydantic` |
| httpx | `httpx` |
| joblib | `joblib` |
| pandas | `pandas` |
| numpy | `numpy` |

**Install all at once:**
```bash
pip install fastapi "uvicorn[standard]" pydantic httpx joblib pandas numpy
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


In [1]:
import subprocess, sys

required = {
    'fastapi': 'fastapi',
    'uvicorn': 'uvicorn[standard]',
    'pydantic': 'pydantic',
    'httpx': 'httpx',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
}
for pkg, inst in required.items():
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', inst, '--quiet', '--break-system-packages'])
print("All packages ready ✅")

All packages ready ✅


## 1. Load the Serialized Model

We load the joblib model produced in Lab 3.1. If not found, we train a fallback.

> **Instructor Note:** The API needs the model to be loaded once at startup, not on every request. Loading at module level (outside the endpoint functions) is the correct pattern — it's called "startup state".


In [2]:
import os, json, warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')

MODEL_PATH = os.path.join('.', 'model_joblib.pkl')
CARD_PATH  = os.path.join('.', 'model_card_v2.json')

# Fallback: also check Module_2
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = os.path.join('..', 'Module_2', 'best_tuned_model.pkl')

try:
    model = joblib.load(MODEL_PATH)
    MODEL_SOURCE = MODEL_PATH
    print(f"✅ Model loaded from: {MODEL_PATH}")
except FileNotFoundError:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    np.random.seed(42)
    n = 800
    FEATURE_NAMES = [
        'cpu_percent','memory_usage_gb','disk_io_mbps',
        'network_rx_mbps','network_tx_mbps','active_vms',
        'stargate_ops','cerebro_replication_lag_s'
    ]
    X_syn = pd.DataFrame(np.random.rand(n, len(FEATURE_NAMES)), columns=FEATURE_NAMES)
    X_syn['cpu_percent'] *= 100
    X_syn['memory_usage_gb'] *= 64
    X_syn['disk_io_mbps'] *= 500
    X_syn['active_vms'] = (X_syn['active_vms'] * 40).astype(int)
    X_syn['stargate_ops'] = (X_syn['stargate_ops'] * 5000).astype(int)
    y_syn = (X_syn['cpu_percent'] > 80).astype(int)
    model = Pipeline([('scaler', StandardScaler()),
                      ('clf', RandomForestClassifier(n_estimators=50, random_state=42))])
    model.fit(X_syn, y_syn)
    MODEL_SOURCE = 'synthetic fallback'
    print("⚠️  Model not found — using synthetic fallback")

# Load model card for feature names
if os.path.exists(CARD_PATH):
    with open(CARD_PATH) as f:
        card = json.load(f)
    FEATURE_NAMES = card['features']
else:
    # Derive feature list from the model if possible
    try:
        FEATURE_NAMES = model.feature_names_in_.tolist()
    except AttributeError:
        FEATURE_NAMES = [
            'cpu_percent','memory_usage_gb','disk_io_mbps',
            'network_rx_mbps','network_tx_mbps','active_vms',
            'stargate_ops','cerebro_replication_lag_s'
        ]

print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES[:5]} ...")


✅ Model loaded from: ./model_joblib.pkl
Features (33): ['cpu_percent', 'memory_mb', 'disk_io_mbps', 'response_time_ms', 'hour_of_day'] ...


## 2. Define Pydantic Schemas

> **Instructor Note:** Pydantic schemas are the contract between the API client and the model. They automatically validate types, ranges, and required fields — and they generate the Swagger UI documentation. Notice we add `Field(ge=0, le=100)` to `cpu_percent` — the API will reject a value of `-5` before it ever reaches the model.


In [3]:
from pydantic import BaseModel, Field
from typing import List, Optional

class CVMFeatures(BaseModel):
    """Input features for a single Nutanix CVM."""
    cpu_percent:                float = Field(..., ge=0, le=100,   description="CPU utilization %")
    memory_usage_gb:            float = Field(..., ge=0,           description="Memory used (GB)")
    disk_io_mbps:               float = Field(..., ge=0,           description="Disk I/O throughput (MB/s)")
    network_rx_mbps:            float = Field(..., ge=0,           description="Network receive (MB/s)")
    network_tx_mbps:            float = Field(..., ge=0,           description="Network transmit (MB/s)")
    active_vms:                 int   = Field(..., ge=0,           description="Number of active VMs")
    stargate_ops:               int   = Field(..., ge=0,           description="Stargate I/O operations/s")
    cerebro_replication_lag_s:  float = Field(..., ge=0,           description="Cerebro replication lag (s)")

    class Config:
        json_schema_extra = {
            "example": {
                "cpu_percent": 87.5,
                "memory_usage_gb": 52.3,
                "disk_io_mbps": 420.1,
                "network_rx_mbps": 95.0,
                "network_tx_mbps": 45.0,
                "active_vms": 28,
                "stargate_ops": 3800,
                "cerebro_replication_lag_s": 12.5,
            }
        }

class PredictionResponse(BaseModel):
    host_id:        str
    is_anomaly:     int
    anomaly_prob:   float
    label:          str
    model_version:  str

class BatchRequest(BaseModel):
    instances: List[CVMFeatures]

class BatchResponse(BaseModel):
    predictions: List[PredictionResponse]
    n_anomalies: int
    anomaly_rate: float

print("✅ Pydantic schemas defined")
print(f"CVMFeatures fields: {list(CVMFeatures.model_fields.keys())}")


✅ Pydantic schemas defined
CVMFeatures fields: ['cpu_percent', 'memory_usage_gb', 'disk_io_mbps', 'network_rx_mbps', 'network_tx_mbps', 'active_vms', 'stargate_ops', 'cerebro_replication_lag_s']


## 3. Build the FastAPI Application

> **Instructor Note:** We define the FastAPI `app` object with all routes. Notice the application does NOT import this notebook — it gets the `model` from a Python module. For testing inside the notebook we pass the model via a global variable. In production the app would live in `app.py` and load the model from disk at startup.


In [4]:
from fastapi import FastAPI, HTTPException
from datetime import datetime
import uuid as _uuid

# ── FastAPI app ───────────────────────────────────────────────────────────
app = FastAPI(
    title="Nutanix Anomaly Detector API",
    description="Predicts whether a CVM is in an anomalous state based on infrastructure telemetry.",
    version="1.0.0",
    docs_url="/docs",
)

MODEL_VERSION = "1.0.0"
START_TIME = datetime.utcnow()

# ── Health check ──────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status":  "healthy",
        "model":   MODEL_VERSION,
        "uptime_s": (datetime.utcnow() - START_TIME).seconds,
    }

# ── Single prediction ─────────────────────────────────────────────────────
@app.post("/predict", response_model=PredictionResponse)
def predict(features: CVMFeatures):
    row = pd.DataFrame([features.model_dump()])

    # Align columns to training feature order; fill missing with 0
    for col in FEATURE_NAMES:
        if col not in row.columns:
            row[col] = 0.0
    row = row[FEATURE_NAMES]

    try:
        pred  = int(model.predict(row)[0])
        prob  = float(model.predict_proba(row)[0][1])
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Model inference failed: {str(e)}")

    return PredictionResponse(
        host_id       = f"ntnx-cvm-{_uuid.uuid4().hex[:6]}",
        is_anomaly    = pred,
        anomaly_prob  = round(prob, 4),
        label         = "ANOMALY" if pred == 1 else "NORMAL",
        model_version = MODEL_VERSION,
    )

# ── Batch prediction ──────────────────────────────────────────────────────
@app.post("/predict/batch", response_model=BatchResponse)
def predict_batch(request: BatchRequest):
    if len(request.instances) > 1000:
        raise HTTPException(status_code=400, detail="Batch size must be ≤ 1000")

    rows = pd.DataFrame([i.model_dump() for i in request.instances])
    for col in FEATURE_NAMES:
        if col not in rows.columns:
            rows[col] = 0.0
    rows = rows[FEATURE_NAMES]

    try:
        preds = model.predict(rows).tolist()
        probs = model.predict_proba(rows)[:, 1].tolist()
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Batch inference failed: {str(e)}")

    predictions = [
        PredictionResponse(
            host_id       = f"ntnx-cvm-{i:03d}",
            is_anomaly    = int(p),
            anomaly_prob  = round(prob, 4),
            label         = "ANOMALY" if p == 1 else "NORMAL",
            model_version = MODEL_VERSION,
        )
        for i, (p, prob) in enumerate(zip(preds, probs))
    ]

    n_anomalies = sum(p.is_anomaly for p in predictions)
    return BatchResponse(
        predictions  = predictions,
        n_anomalies  = n_anomalies,
        anomaly_rate = round(n_anomalies / len(predictions), 4),
    )

print("✅ FastAPI app built with routes: /health  /predict  /predict/batch")


✅ FastAPI app built with routes: /health  /predict  /predict/batch


## 4. Test with TestClient

`TestClient` runs the ASGI app in-process — no server, no ports, no network. It's the standard way to unit-test FastAPI applications.

> **Instructor Note:** Show students that `TestClient` catches validation errors before they reach the model. This is exactly how the CI test suite for a production API would work.


In [5]:
from fastapi.testclient import TestClient

client = TestClient(app)

# ── Test /health ──────────────────────────────────────────────────────────
resp = client.get("/health")
print(f"GET /health  →  {resp.status_code}")
print(json.dumps(resp.json(), indent=2))


GET /health  →  200
{
  "status": "healthy",
  "model": "1.0.0",
  "uptime_s": 0
}


In [6]:
# ── Test /predict — normal CVM ───────────────────────────────────────────
normal_payload = {
    "cpu_percent": 32.1,
    "memory_usage_gb": 18.0,
    "disk_io_mbps": 120.5,
    "network_rx_mbps": 40.0,
    "network_tx_mbps": 22.0,
    "active_vms": 12,
    "stargate_ops": 1200,
    "cerebro_replication_lag_s": 2.1,
}
resp = client.post("/predict", json=normal_payload)
print(f"POST /predict (normal CVM)  →  {resp.status_code}")
print(json.dumps(resp.json(), indent=2))


POST /predict (normal CVM)  →  200
{
  "host_id": "ntnx-cvm-500e26",
  "is_anomaly": 0,
  "anomaly_prob": 0.0064,
  "label": "NORMAL",
  "model_version": "1.0.0"
}


In [7]:
# ── Test /predict — anomalous CVM ────────────────────────────────────────
anomaly_payload = {
    "cpu_percent": 96.5,
    "memory_usage_gb": 61.0,
    "disk_io_mbps": 490.0,
    "network_rx_mbps": 185.0,
    "network_tx_mbps": 170.0,
    "active_vms": 38,
    "stargate_ops": 4800,
    "cerebro_replication_lag_s": 95.0,
}
resp = client.post("/predict", json=anomaly_payload)
print(f"POST /predict (anomalous CVM)  →  {resp.status_code}")
print(json.dumps(resp.json(), indent=2))


POST /predict (anomalous CVM)  →  200
{
  "host_id": "ntnx-cvm-6d1b97",
  "is_anomaly": 0,
  "anomaly_prob": 0.2554,
  "label": "NORMAL",
  "model_version": "1.0.0"
}


In [8]:
# ── Test validation — out-of-range cpu_percent ───────────────────────────
bad_payload = {**normal_payload, "cpu_percent": 150.0}  # > 100 is invalid
resp = client.post("/predict", json=bad_payload)
print(f"POST /predict (invalid cpu=150)  →  {resp.status_code}  (expected 422)")
print(json.dumps(resp.json()['detail'][0], indent=2))


POST /predict (invalid cpu=150)  →  422  (expected 422)
{
  "type": "less_than_equal",
  "loc": [
    "body",
    "cpu_percent"
  ],
  "msg": "Input should be less than or equal to 100",
  "input": 150.0,
  "ctx": {
    "le": 100.0
  }
}


In [9]:
# ── Test /predict/batch — 5 CVMs ─────────────────────────────────────────
np.random.seed(7)
batch_instances = []
for i in range(5):
    batch_instances.append({
        "cpu_percent":               float(np.random.uniform(20, 99)),
        "memory_usage_gb":           float(np.random.uniform(8, 62)),
        "disk_io_mbps":              float(np.random.uniform(50, 490)),
        "network_rx_mbps":           float(np.random.uniform(10, 180)),
        "network_tx_mbps":           float(np.random.uniform(5, 160)),
        "active_vms":                int(np.random.randint(2, 40)),
        "stargate_ops":              int(np.random.randint(200, 4900)),
        "cerebro_replication_lag_s": float(np.random.uniform(0, 100)),
    })

resp = client.post("/predict/batch", json={"instances": batch_instances})
print(f"POST /predict/batch (5 CVMs)  →  {resp.status_code}")
result = resp.json()
print(f"Anomalies: {result['n_anomalies']}/{len(batch_instances)}  "
      f"(rate={result['anomaly_rate']:.1%})")
print("\nPer-CVM results:")
for p in result['predictions']:
    bar = "🔴" if p['is_anomaly'] else "🟢"
    print(f"  {bar}  {p['host_id']:20s}  prob={p['anomaly_prob']:.3f}  {p['label']}")


POST /predict/batch (5 CVMs)  →  200
Anomalies: 0/5  (rate=0.0%)

Per-CVM results:
  🟢  ntnx-cvm-000          prob=0.006  NORMAL
  🟢  ntnx-cvm-001          prob=0.006  NORMAL
  🟢  ntnx-cvm-002          prob=0.244  NORMAL
  🟢  ntnx-cvm-003          prob=0.006  NORMAL
  🟢  ntnx-cvm-004          prob=0.007  NORMAL


## 5. Write the Production app.py

> **Instructor Note:** Notebooks are not deployable — they require a kernel. The production service is a plain Python file. We write `app.py` to disk here so students see exactly what transitions from the notebook to the file.


In [10]:
app_code = '''"""
Nutanix Anomaly Detector — FastAPI service
Usage:
    uvicorn app:app --host 0.0.0.0 --port 8000 --reload
"""
import os, json
import numpy as np
import pandas as pd
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime
import uuid as _uuid

MODEL_PATH   = os.getenv("MODEL_PATH",  "model_joblib.pkl")
CARD_PATH    = os.getenv("CARD_PATH",   "model_card_v2.json")
MODEL_VERSION = "1.0.0"
START_TIME   = datetime.utcnow()

model = joblib.load(MODEL_PATH)
with open(CARD_PATH) as f:
    card = json.load(f)
FEATURE_NAMES = card["features"]

app = FastAPI(title="Nutanix Anomaly Detector API", version=MODEL_VERSION)

class CVMFeatures(BaseModel):
    cpu_percent:               float = Field(..., ge=0, le=100)
    memory_usage_gb:           float = Field(..., ge=0)
    disk_io_mbps:              float = Field(..., ge=0)
    network_rx_mbps:           float = Field(..., ge=0)
    network_tx_mbps:           float = Field(..., ge=0)
    active_vms:                int   = Field(..., ge=0)
    stargate_ops:              int   = Field(..., ge=0)
    cerebro_replication_lag_s: float = Field(..., ge=0)

class PredictionResponse(BaseModel):
    host_id: str; is_anomaly: int; anomaly_prob: float; label: str; model_version: str

class BatchRequest(BaseModel):
    instances: List[CVMFeatures]

class BatchResponse(BaseModel):
    predictions: List[PredictionResponse]; n_anomalies: int; anomaly_rate: float

@app.get("/health")
def health():
    return {"status": "healthy", "model": MODEL_VERSION,
            "uptime_s": (datetime.utcnow() - START_TIME).seconds}

@app.post("/predict", response_model=PredictionResponse)
def predict(features: CVMFeatures):
    row = pd.DataFrame([features.model_dump()])[FEATURE_NAMES]
    pred = int(model.predict(row)[0])
    prob = float(model.predict_proba(row)[0][1])
    return PredictionResponse(host_id=f"ntnx-cvm-{_uuid.uuid4().hex[:6]}",
        is_anomaly=pred, anomaly_prob=round(prob,4),
        label="ANOMALY" if pred==1 else "NORMAL", model_version=MODEL_VERSION)

@app.post("/predict/batch", response_model=BatchResponse)
def predict_batch(request: BatchRequest):
    rows  = pd.DataFrame([i.model_dump() for i in request.instances])[FEATURE_NAMES]
    preds = model.predict(rows).tolist()
    probs = model.predict_proba(rows)[:,1].tolist()
    out   = [PredictionResponse(host_id=f"ntnx-cvm-{i:03d}", is_anomaly=int(p),
             anomaly_prob=round(pr,4), label="ANOMALY" if p==1 else "NORMAL",
             model_version=MODEL_VERSION) for i,(p,pr) in enumerate(zip(preds,probs))]
    return BatchResponse(predictions=out, n_anomalies=sum(p.is_anomaly for p in out),
                         anomaly_rate=round(sum(p.is_anomaly for p in out)/len(out),4))
'''

app_path = os.path.join('.', 'app.py')
with open(app_path, 'w') as f:
    f.write(app_code.strip())

print(f"✅ app.py written to {os.path.abspath(app_path)}")
print("\nTo run the service locally:")
print("  source ~/aiml-venv/bin/activate")
print("  cd Module_3")
print("  uvicorn app:app --host 0.0.0.0 --port 8000 --reload")
print("\nThen open:  http://localhost:8000/docs")


✅ app.py written to /Users/nikhil/AI:ML intermediate/Module_3/app.py

To run the service locally:
  source ~/aiml-venv/bin/activate
  cd Module_3
  uvicorn app:app --host 0.0.0.0 --port 8000 --reload

Then open:  http://localhost:8000/docs


## 6. Live API Test (Optional — requires uvicorn running)

Run this cell only after starting the server in a terminal with:
```bash
source ~/aiml-venv/bin/activate
cd Module_3
uvicorn app:app --port 8000 --reload
```

> **Instructor Note:** Walk students through the Swagger UI at `http://localhost:8000/docs`. The schemas, examples, and validation rules are all auto-generated from the Pydantic models — no documentation to write separately.


In [11]:
import httpx

# Set LIVE_TEST = True only if uvicorn is running on port 8000
LIVE_TEST = False

if LIVE_TEST:
    base = "http://localhost:8000"
    r = httpx.get(f"{base}/health")
    print(f"Health: {r.json()}")

    r = httpx.post(f"{base}/predict", json=anomaly_payload)
    print(f"Live predict: {r.json()}")
else:
    print("ℹ️  LIVE_TEST is False — using TestClient results above.")
    print("   Set LIVE_TEST = True and start uvicorn to test the live server.")


ℹ️  LIVE_TEST is False — using TestClient results above.
   Set LIVE_TEST = True and start uvicorn to test the live server.


## Lab Summary

- **FastAPI** provides automatic input validation (Pydantic), serialization, and Swagger docs  
- **TestClient** lets you test the entire API in-notebook without starting a server  
- **`app.py`** is the production artifact — notebooks train, Python files serve  
- Next step: containerise this service with Docker (Lab 3.3)

---
## 🎯 Challenges

### Challenge 1
Add a new endpoint `GET /model/info` that returns the model metadata from `model_card_v2.json` (feature list, version, file sizes). Test it with `TestClient`.


In [12]:
# Challenge 1 — your code here


### Challenge 2
The `/predict` endpoint generates a random `host_id`. Modify it to accept an optional `host_id` field in the request body (e.g., `"ntnx-cvm-007"`). If the caller provides it, use it; if not, generate a UUID-based one. Update the Pydantic schema and test both cases.


In [13]:
# Challenge 2 — your code here


### Challenge 3
Add a `/predict/batch` size limit test. Send a batch with 1001 instances (duplicate the existing 5-instance batch ~200 times). Verify the API returns `HTTP 400` with the correct error message.


In [14]:
# Challenge 3 — your code here
